# N3 — Activation Function Comparison

**Supplementary experiment: do smooth activations shift the double-descent peak?**

| | |
|---|---|
| Model | CNN5 (5-layer CNN, width multiplier k) |
| Dataset | CIFAR-10, n=5 000, fixed η=15% |
| Sweep | k ∈ {1,2,3,4,6,8,16,32} × act ∈ {ReLU, GELU, Tanh} × 1 seed = **24 runs** |
| Owner | Lyric (D) |
| Output | Fig 7 — activation comparison curves, n3_summary.csv |

**Motivation:** Under the Neural Tangent Kernel (NTK) framework, smooth activations
(GELU, Tanh) have different spectral properties from the non-smooth ReLU. This may
shift the position or height of the interpolation peak in the double-descent curve.
Even a null result (no significant shift) is a valuable finding for the Discussion section.

## Step 1 — Environment

In [ ]:
!pip install -q torch torchvision numpy matplotlib seaborn tqdm pandas
import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 2 — Mount Drive + Clone repo

In [ ]:
import os, sys
from google.colab import drive

# ── Config ───────────────────────────────────────────────────────────────────
TOKEN      = 'ghp_你的token'    # ← replace with your PAT
REPO_DIR   = '/content/project-6699'
RESULT_DIR = '/content/drive/MyDrive/benign_overfitting/N3'

# ── Mount Drive ───────────────────────────────────────────────────────────────
drive.mount('/content/drive')
os.makedirs(RESULT_DIR, exist_ok=True)
print(f'Results → {RESULT_DIR}')

# ── Clone / update repo ───────────────────────────────────────────────────────
REPO_URL = f'https://{TOKEN}@github.com/alice20030504/EECS-6699.git'
if not os.path.exists(REPO_DIR):
    os.system(f'git clone --branch main {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} checkout main')
    os.system(f'git -C {REPO_DIR} pull')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Files:', [f for f in os.listdir('.') if f.endswith('.py')])

## Step 3 — Config

N3 is a single-account experiment (**24 runs total**, ~3 h on a T4 GPU).  
No parallelisation needed — just run all cells.

> **Resuming from first batch?** If you already ran k={4,8,16,32}, those 12
> results are auto-skipped. Only the 12 new peak-region runs (k=1,2,3,6) will
> be trained (~1.5 h). Results from both batches are combined automatically.

In [ ]:
from run_n3 import N3_CONFIG, run_n3, plot_n3
import copy

cfg = copy.deepcopy(N3_CONFIG)

# Optional: run a subset of activations or widths
# MY_ACTIVATIONS = ['relu']          # quick smoke test (1 activation)
# MY_WIDTHS = [1, 2, 3, 6]          # peak-region supplement only
MY_ACTIVATIONS = None                # all three (default)
MY_WIDTHS      = None                # all eight widths (default)

active_acts   = MY_ACTIVATIONS if MY_ACTIVATIONS else cfg['activations']
active_widths = MY_WIDTHS      if MY_WIDTHS      else cfg['widths']
n_runs        = len(active_widths) * len(active_acts) * len(cfg['seeds'])
print(f"Activations : {active_acts}")
print(f"Widths      : {active_widths}")
print(f"Fixed noise : {cfg['noise_rate']:.0%}")
print(f"Total runs  : {n_runs}  (completed runs will be auto-skipped)")

## Step 4 — Run N3

In [ ]:
# Keep-alive thread (prevents Colab from disconnecting during long runs)
import threading, time
def _keep_alive():
    while True:
        time.sleep(60)
        try:
            from google.colab.output import eval_js
            eval_js('0')
        except Exception:
            pass
threading.Thread(target=_keep_alive, daemon=True).start()

results = run_n3(cfg, RESULT_DIR, widths=MY_WIDTHS, activations=MY_ACTIVATIONS, resume=True)
print(f'\nDone. {len(results)} runs saved to {RESULT_DIR}')

## Step 5 — Plot (Fig 7)

In [ ]:
plot_n3(RESULT_DIR)

from IPython.display import Image, display
from pathlib import Path

fig_path = Path(RESULT_DIR) / 'fig7_n3_activation.png'
if fig_path.exists():
    print('--- fig7_n3_activation.png ---')
    display(Image(str(fig_path)))
else:
    print('[warn] Figure not found. Did the run complete?')

## Step 6 — Summary table

In [ ]:
import pandas as pd
from src.io_utils import load_results
from pathlib import Path

results = load_results(RESULT_DIR, pattern='n3_*.json')
df = pd.DataFrame([{
    'activation': r['activation'],
    'k':          r['width_multiplier'],
    'seed':       r['seed'],
    'n_params':   r.get('n_params', '—'),
    'train_err':  f"{r['train_error']:.3f}",
    'test_err':   f"{r['test_error']:.3f}",
} for r in results]).sort_values(['activation', 'k'])

print(df.to_string(index=False))

## Step 7 — Analysis notes

Key observations to check and report in the paper (Sec 4.3):

| Question | What to look for in Fig 7 |
|----------|---------------------------|
| Peak position | Does GELU/Tanh peak at a different k than ReLU? |
| Peak height | Is the maximum test error higher or lower for smooth activations? |
| Curve shape | Are smooth activation curves flatter (less pronounced DD)? |
| Benign region | At large k, do smooth activations reach lower test error? |

Even a **null result** (all three curves nearly identical) is publishable —  
it suggests the double-descent phenomenon is activation-agnostic for CNN5 on CIFAR-10  
and can be discussed in the Discussion / Conclusion section.